# SQL-Analyst Agent — Colab Training Notebook

Reproduces this repo's training runs end to end: clones the repo, installs dependencies, then walks through the SFT arm (`collect_sft_data.py` → `train_sft.py` → held-out eval) and the GRPO arm (`train_grpo.py` → `evaluate_grpo.py`).

**Before running anything:**
1. `Runtime > Change runtime type` → pick a GPU (T4 or better).
2. Add a Colab secret named exactly `WANDB_API_KEY` (key icon, left sidebar) with your Weights & Biases API key.
3. Optional but recommended: add a Colab secret named `HF_TOKEN` (a [Hugging Face token](https://huggingface.co/settings/tokens), read access is enough) - without it, model/dataset downloads are rate-limited harder and can occasionally hit `429 Too Many Requests`.

**One interruption:** after the install cells, you'll be asked to restart the runtime once. This is expected and normal. See the markdown cell where it happens for why. Everything before that point only needs to run once; after restarting, skip straight past it to the cell right after.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected. Runtime > Change runtime type > pick a GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

REPO_URL = "https://github.com/AryamanJaggi/sql-agent-rlvr.git"
REPO_DIR = "/content/sql-agent-rlvr"

# Absolute paths throughout, so this cell is safe to re-run from any cwd -
# a relative `%cd sql-agent-rlvr` re-run from inside an already-nested
# checkout would clone and cd one level deeper every time.
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q --retries 5 --timeout 120 -r requirements.txt
!pip install -q --retries 5 --timeout 120 unsloth vllm trl wandb

In [ ]:
# Unsloth checks the installed vLLM build against this runtime's CUDA
# version and blocks the import if they don't match.
!pip install -q --force-reinstall --no-deps https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl

## Restart runtime now

**Runtime → Restart session**

This is required because Unsloth's import-compatibility check patches Python's import machinery for the rest of the process once it runs, so simply having reinstalled the correct vLLM build above isn't enough within the same running kernel. The block has to be cleared by actually restarting.

**After restarting, continue from the next cell below.** Do not re-run the cells above (clone/install).

In [ ]:
%cd /content/sql-agent-rlvr

import torch
from unsloth import FastLanguageModel
from vllm import SamplingParams
print("Imports OK - vLLM/CUDA mismatch is cleared.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = "/content/drive/MyDrive/sql_agent_rlvr_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results will be saved under:", RESULTS_DIR)

# Higher HF Hub rate limit + fewer 429s on model/dataset downloads.
# transformers/datasets/huggingface_hub all pick this up automatically.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN set from Colab secret.")
except Exception:
    print("No HF_TOKEN secret found - continuing unauthenticated (fine, just slower/stricter rate limits).")

## SFT: collect_sft_data.py → train_sft.py → eval

Runs the untrained prompted baseline over Spider's `train` split (hard/extra difficulty tiers), keeps only its successful trajectories, fine-tunes a LoRA adapter on them, then evaluates that adapter on the `validation` split through the same ReAct harness the baseline was measured with.

Each step below runs a small test first to ensure there are no errors, then the real run.

In [ ]:
!python -m train.collect_sft_data --limit 3 --output {RESULTS_DIR}/sft_data_smoke.jsonl

In [ ]:
!python -m train.collect_sft_data --limit 150 --output {RESULTS_DIR}/sft_data.jsonl

In [ ]:
!python -m train.train_sft --data {RESULTS_DIR}/sft_data_smoke.jsonl --output-dir {RESULTS_DIR}/sft_adapter_smoke --epochs 1

In [ ]:
!python -m train.train_sft --data {RESULTS_DIR}/sft_data.jsonl --output-dir {RESULTS_DIR}/sft_adapter

In [ ]:
!python -m eval.evaluate --policy unsloth --lora-path {RESULTS_DIR}/sft_adapter --split validation --limit 30 --wandb-project sql-agent-rlvr

## GRPO arm: train_grpo.py → evaluate_grpo.py

Trains via TRL's `environment_factory` (native tool-calling), then evaluates with a matching native-tool-calling harness.

**This section alternates between two runtimes.** `train_grpo.py` needs `trl>=1.0` for `environment_factory`; every other cell in this notebook needs Unsloth, which caps `trl<=0.24.0`. The two can't coexist in one environment, so training and evaluation run in separate runtimes. Each step below says which one it needs.

The order is *train smoke test → eval smoke test + headroom → real training*, not train-then-eval. Headroom calibration gates whether the real run is worth starting at all (see step 3), and it needs the chat template that a training run writes out — so the cheap smoke run comes first.

### Step 1 — fresh runtime, no Unsloth: install + training smoke test

**Runtime → Disconnect and delete runtime**, then **Runtime → Change runtime type** and pick a GPU. In the fresh runtime, scroll up and re-run only the **clone/pull cell** and the **Drive-mount cell** — skip the Unsloth install and the restart cells entirely.

Then run the install cell below, then the smoke test. The smoke run takes a few minutes and exists to prove the pipeline end to end; it also writes the adapter directory whose tokenizer step 2 needs.

In [ ]:
# Separate from the rest of this notebook: no unsloth here, since
# environment_factory needs trl>=1.0 and Unsloth caps trl<=0.24.0.
# trl is pinned exactly: train_grpo.py depends on version-specific
# behaviour (trl.chat_template_utils, environment_factory's contract,
# GRPOConfig field names), all verified against 1.10.0.
# transformers must be pinned too: trl 1.10's environment_factory hard-
# requires >=5.2.0, and >=5.13 additionally enables the new-style
# response_template path it uses to parse tool calls back out of a
# generation. Left unpinned, pip resolves a 4.x build on some runs and
# GRPOTrainer raises ImportError before training starts.
# gguf: vllm's config loader imports it unconditionally (GGUF model
# support) even though we never load a GGUF model - not pulled in as
# a hard vllm dependency, so it needs installing explicitly. vllm
# itself is required even though training runs with use_vllm=False,
# because trl.trainer.grpo_trainer imports it at module load.
# torchao: Colab's base image ships 0.10.0, but peft's LoRA dispatcher
# requires >=0.16.0 to even probe whether torchao applies - an old
# (not missing) version makes it raise instead of skipping cleanly,
# even though we never use torchao-quantized models.
!pip install -q --retries 5 --timeout 120 -r requirements.txt
!pip install -q --retries 5 --timeout 120 torch "trl==1.10.0" "transformers>=5.13,<6" peft bitsandbytes vllm gguf "torchao>=0.16.0" wandb
!pip install -q --force-reinstall --no-deps https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl
import transformers, trl; print("transformers", transformers.__version__, "| trl", trl.__version__)

In [ ]:
!python -m train.train_grpo --limit 5 --num-generations 2 --epochs 1 --output-dir {RESULTS_DIR}/grpo_adapter_smoke

### Step 2 — back to an Unsloth runtime: eval smoke test + headroom calibration

`evaluate_grpo.py` drives tool-calling itself and never touches `GRPOTrainer`, so it runs on the normal Unsloth stack. **Runtime → Disconnect and delete runtime**, start a fresh one, and run the cells from the top of this notebook (GPU check → clone/pull → installs → restart → import check → Drive mount).

The first cell below is a 2-example smoke test: it's the first time `parse_tool_call` meets real model output, so check the printed transcript actually shows tool calls being parsed rather than silently failing.

The second cell is the **headroom measurement**, and it's the one that decides whether the real training run is worth starting.

In [ ]:
!python -m eval.evaluate_grpo --lora-path {RESULTS_DIR}/grpo_adapter_smoke --split validation --limit 2 --max-steps 5

In [ ]:
# Untrained base model under the GRPO protocol: no --lora-path, so no adapter
# weights are loaded. --tokenizer-path still points at a train_grpo.py output
# dir so prompts render with the same chat template training uses - measuring
# against the base repo's template would not be the thing GRPO actually trains.
#
# Headroom rule: the base has to land roughly in the 20-60% band, or GRPO's
# group-relative advantage is zero everywhere and there is no gradient.
# The earlier ReAct calibration does NOT transfer - native tool-calling is a
# different protocol and empirically a harder one for this model.
!python -m eval.evaluate_grpo --tokenizer-path {RESULTS_DIR}/grpo_adapter_smoke --split validation --limit 30 --difficulty hard
!python -m eval.evaluate_grpo --tokenizer-path {RESULTS_DIR}/grpo_adapter_smoke --split validation --limit 30 --difficulty extra

### Step 3 — read the headroom numbers before spending GPU hours

- **Both tiers inside ~20-60%:** proceed as-is.
- **Well below 20%** (all-zero groups, no gradient): the task is too hard under this protocol. Add an easier tier by editing `DIFFICULTY_TIERS` in `train/train_grpo.py` — it's a module constant rather than a CLI flag on purpose, so the tier mix is a recorded decision and not an invisible per-run knob. Record the change alongside the original calibration notes.
- **Above 60%:** drop the easier tier, for the same reason in reverse.

Also size the run before launching it. Cost is `examples × num_generations` completions; time each from the smoke run's `step_time` and completion count. Shrinking `--num-generations` is the wrong lever — small groups are exactly what collapses the reward variance GRPO learns from — so trade `--limit` down instead. Raising `--batch-size`/`--grad-accum` puts more sequences in flight per generation call and can cut wall-clock without changing total work, as long as their product stays divisible by `--num-generations`.

### Step 4 — the real training run (fresh non-Unsloth runtime again)

Same runtime switch as step 1: delete the runtime, start a GPU one, re-run the clone/pull and Drive-mount cells, then the step-1 install cell, then this.

**Before re-running this after a failed or unsatisfactory attempt:** `train_grpo.py` auto-resumes from the last checkpoint under `--output-dir` if one exists (see the comment above `trainer.train(...)` in the script). That's exactly what you want after a crash mid-run - but if you're intentionally starting over with different hyperparameters, resuming from a checkpoint that already hit the step target means training does nothing. In Drive, **rename** (don't delete - it's a real result worth keeping for the writeup) the previous `grpo_adapter` folder to something like `grpo_adapter_run1` before running this cell again, so it starts fresh.

In [ ]:
# Run 1 (2026-08-22) results: hard/extra success rate flat vs baseline
# (~50%/17%), but the KL trace spiked to ~4.7e6 around step 70 and
# frac_reward_zero_std climbed 0.5->0.8 over training - entropy
# collapsing to ~0.06 (well under TRL's own entropy_target default of
# 0.2). Changes for run 2, all explained in train_grpo.py:
#   --learning-rate 2e-6 (down from run 1's 5e-6, which was too
#     aggressive for how few groups actually produced gradient)
#   --temperature 1.15 (hotter rollout sampling to fight the collapse
#     directly, on top of use_adaptive_entropy=True which is now
#     unconditionally on in training_hyperparams())
#   --batch-size 2 --grad-accum 8 (down from the 4/4 that ran fine in
#     run 1): use_adaptive_entropy=True made a first attempt at this OOM
#     at 79.2/79.25 GiB, and the traceback lands in
#     entropy_from_logits - adaptive entropy control needs a per-token,
#     full-vocab entropy tensor that run 1 never computed. Effective
#     batch stays 4*4 == 2*8 == 16, still divisible by
#     --num-generations 8, so total optimizer steps and the LR schedule
#     are unchanged. This halves (not quarters) the per-forward-pass
#     sequence count - if it still OOMs, drop --batch-size to 1 and
#     --grad-accum to 16 instead.
# grpo_adapter/ from run 1 is a separate, complete run - nothing here
# resumes from it. See the markdown note above about clearing it out
# first if you don't want to keep run 1's checkpoints on Drive.
!python -m train.train_grpo --limit 150 --learning-rate 2e-6 --temperature 1.15 --batch-size 2 --grad-accum 8 --output-dir {RESULTS_DIR}/grpo_adapter


### Step 5 — final held-out eval (back to an Unsloth runtime)

Same switch as step 2. This is the number that goes in the three-way comparison table.

In [ ]:
!python -m eval.evaluate_grpo --lora-path {RESULTS_DIR}/grpo_adapter --split validation --limit 30